# MLPRegressor (64, 32) 최종 테스트 및 저장 모델 연결 검증

20개 origin 검증과 제한적 튜닝으로 설정을 고정한 뒤 수행한 실제 결과입니다. 코드 셀 재실행은 저장된 결과 조회만 수행하며 학습·공유 모델 덮어쓰기·예측 재실행을 하지 않습니다.

현재 수정치·달력 정렬 데이터이며 과거 공개일·빈티지는 미검증입니다. 역사적 as-of 재현 백테스트를 보장하지 않습니다.

In [1]:
from pathlib import Path
import json,pandas as pd
ROOT=Path.cwd().resolve()
if not (ROOT/'configs').exists():
    ROOT=ROOT.parent
OUT=ROOT/'.private/mlp/mlp_64_32_final_20261110_01'
TUNING=ROOT/'.private/mlp/mlp_64_32_tuning_20261110_01'
RUN_TRAINING=False
print('결과 폴더:',OUT)
print('재학습:',RUN_TRAINING)


결과 폴더: D:\hjs\five-stack\.private\mlp\mlp_64_32_final_20261110_01
재학습: False


## 검증 기반 설정 고정

공통 구조: 바깥 Pipeline → ColumnTransformer(범주형 인코딩, 수치 결측 처리·StandardScaler) → TransformedTargetRegressor(MLPRegressor + 타깃 StandardScaler)입니다. predict는 USD 원금액을 반환합니다.

동일한 검증 h=2~5 정답 9,164행의 WAPE로 선정했습니다. 테스트를 확인한 뒤 파라미터를 변경하지 않았습니다. 포함 학습률 0.0005, 제외 0.001이며 두 조건 모두 alpha=0.0001, max_iter=300입니다. ReLU·Adam·batch_size=256·random_state=42·early_stopping=False·shuffle=False를 유지했습니다.

In [2]:
lock=json.loads((OUT/'selection_lock.json').read_text(encoding='utf-8'))
print(json.dumps(lock,ensure_ascii=False,indent=2))
print(pd.read_csv(TUNING/'all_candidates_comparison.csv')[['candidate','macro_mode','WAPE','MAE','RMSE','R2']].to_string(index=False))


{
  "params": {
    "with_macro": {
      "hidden_layer_sizes": [
        64,
        32
      ],
      "activation": "relu",
      "solver": "adam",
      "learning_rate_init": 0.0005,
      "alpha": 0.0001,
      "batch_size": 256,
      "max_iter": 300,
      "random_state": 42,
      "early_stopping": false,
      "shuffle": false
    },
    "without_macro": {
      "hidden_layer_sizes": [
        64,
        32
      ],
      "activation": "relu",
      "solver": "adam",
      "learning_rate_init": 0.001,
      "alpha": 0.0001,
      "batch_size": 256,
      "max_iter": 300,
      "random_state": 42,
      "early_stopping": false,
      "shuffle": false
    }
  },
  "train_end": "2025Q4",
  "dataset_version": "v002",
  "source": "D:\\hjs\\five-stack\\.private\\mlp\\mlp_64_32_tuning_20261110_01\\selected_configuration.json",
  "source_sha256": "508d78bc26794179996803f523463e4f94089f35804cb89034a7f2573936b857",
  "criterion": "minimum validation h2_5 WAPE on identical 9164 observed 

## 최종 학습과 수렴

두 모델 모두 target_quarter<=2025Q4인 관측 정답 34,745행만 학습했습니다. 전처리와 타깃 표준화도 이 범위에서만 fit했습니다. 손실은 표준화된 타깃 공간의 학습 손실이며 USD 오차가 아닙니다.

In [3]:
for mode in ['with_macro','without_macro']:
    audit=json.loads((OUT/f'fit_audit_{mode}.json').read_text(encoding='utf-8'))
    print(json.dumps(audit,ensure_ascii=False,indent=2))


{
  "macro_mode": "with_macro",
  "fit_seconds": 6.7855159000027925,
  "fit_rows": 34745,
  "train_end": "2025Q4",
  "fit_key_hash": "48af33eb6587a804e3b67a7fe30f6c66c35ba814ced1cf70bbfde768727a501d",
  "status": "converged",
  "converged": true,
  "n_iter": 56,
  "max_iter": 300,
  "loss": 0.023114496031125753,
  "loss_curve": [
    0.1749145573975217,
    0.0637600918172506,
    0.04311811859263638,
    0.039603218429012,
    0.04249587419184678,
    0.04611047273744063,
    0.052580429817269375,
    0.05733145549314087,
    0.07201469651252441,
    0.051484947994069406,
    0.045524972078721505,
    0.03917709097237494,
    0.037541049053816625,
    0.03309806519692512,
    0.03362802511627354,
    0.03782542502916546,
    0.04506711932297523,
    0.043748042150424396,
    0.036879235882199914,
    0.031523502662048494,
    0.030192153365298638,
    0.027303747795455732,
    0.02811812511925817,
    0.028421672510693215,
    0.03367940714804286,
    0.03349775867019369,
    0.032163

## 검증 및 2026년 최종 테스트 성능

검증은 h=2~5, 최종 테스트는 2025Q4 고정 origin의 2026Q1~Q3(h=1~3)입니다. 전체 480행을 예측하고 정답 확보 447행만 평가했습니다. 정답 결측 33행은 보존했습니다. WAPE는 %, MAE·RMSE는 USD, R²는 무차원입니다.

In [4]:
summary=pd.read_csv(OUT/'model_comparison.csv')
print(summary.to_string(index=False))
for mode in ['with_macro','without_macro']:
    pred=pd.read_parquet(OUT/f'final_test_predictions_{mode}.parquet')
    print(mode,'전체=',len(pred),'평가=',int(pred.target_available.sum()),'정답결측=',int(pred.target_export_value_usd.isna().sum()))
    print(pred[['country_code','hs_code','target_quarter','target_export_value_usd','raw_predicted_export_value_usd','predicted_export_value_usd','raw_negative']].head(8).to_string(index=False))


 model_id    macro_mode                                                                                                                                                                                                            hyperparameters dataset_version train_end  feature_count  validation_WAPE  validation_MAE  validation_RMSE  validation_R2  test_WAPE     test_MAE    test_RMSE  test_R2  n_predictions  n_evaluated  n_missing_targets  raw_negative_count  raw_negative_evaluated  postprocessed_negative_count  nonfinite_count                                              evaluation_key_hash  usd_inverse_exact                                                      model_path  historical_asof_verified
mlp_64_32    with_macro {"activation": "relu", "alpha": 0.0001, "batch_size": 256, "early_stopping": false, "hidden_layer_sizes": [64, 32], "learning_rate_init": 0.0005, "max_iter": 300, "random_state": 42, "shuffle": false, "solver": "adam"}            v002    2025Q4             57        42

## joblib 저장 및 새 Python 프로세스 재로딩

공통 사양의 dict에 학습 완료 Pipeline·입력 순서·범주형 목록·v002·학습 종료 2025Q4를 저장했습니다. 모델 폴더에는 pipeline.joblib만 있습니다. 별도 Python 프로세스에서 공통 load_model로 읽고 fit을 금지한 상태에서 480행의 원예측·후처리 예측 완전 일치를 확인했습니다.

In [5]:
print(summary[['macro_mode','model_path']].to_string(index=False))
print((OUT/'new_process_reload_verification.json').read_text(encoding='utf-8'))


   macro_mode                                                      model_path
   with_macro    artifacts\shared_models\mlp_64_32\with_macro\pipeline.joblib
without_macro artifacts\shared_models\mlp_64_32\without_macro\pipeline.joblib
[
  {
    "macro_mode": "with_macro",
    "new_process": true,
    "rows": 480,
    "raw_exact": true,
    "clipped_exact": true,
    "max_absolute_difference": 0.0,
    "fit_forbidden": true
  },
  {
    "macro_mode": "without_macro",
    "new_process": true,
    "rows": 480,
    "raw_exact": true,
    "clipped_exact": true,
    "max_absolute_difference": 0.0,
    "fit_forbidden": true
  }
]


## 2027년 공통 엔진 연결 점검

기존 scripts/forecast.py --model mlp_64_32로 두 거시 조건을 각각 실행했습니다. 2025Q4 테스트용 모델에 2026Q3 입력을 제공한 연결 점검입니다. 2026Q3까지 재학습한 운영용 모델이나 2027년 정확도 평가가 아닙니다. 2027년 정답 평가·운영용 재학습은 수행하지 않았습니다.

In [6]:
print((OUT/'forecast_verification.json').read_text(encoding='utf-8'))
print((OUT/'status.json').read_text(encoding='utf-8'))


[
  {
    "macro_mode": "with_macro",
    "status": "passed",
    "rows": 640,
    "annual_rows": 160,
    "countries": 20,
    "hs6": 8,
    "quarters": 4,
    "duplicate_keys": 0,
    "nonfinite_count": 0,
    "missing_predictions": 0,
    "postprocessed_negative_count": 0,
    "raw_negative_count": 261,
    "refit_performed": false,
    "model_train_end": "2025Q4",
    "prediction_origin": "2026Q3",
    "connection_check_only": true,
    "operational_model": false,
    "accuracy_2027_evaluated": false,
    "cli_returncode": 0
  },
  {
    "macro_mode": "without_macro",
    "status": "passed",
    "rows": 640,
    "annual_rows": 160,
    "countries": 20,
    "hs6": 8,
    "quarters": 4,
    "duplicate_keys": 0,
    "nonfinite_count": 0,
    "missing_predictions": 0,
    "postprocessed_negative_count": 0,
    "raw_negative_count": 366,
    "refit_performed": false,
    "model_train_end": "2025Q4",
    "prediction_origin": "2026Q3",
    "connection_check_only": true,
    "operational_m

## 환경 및 보존 기록

기존 데이터·공통 엔진·다른 모델·기존 검증·튜닝 결과·패키지 버전은 유지했습니다. Git commit·push·merge는 수행하지 않았습니다. 개인 평가 파일은 .private/mlp/에 보관하며 별도 공유하지 않으면 다른 환경에서 결과 조회 시 해당 파일을 찾을 수 없습니다. 초기 소규모 점검 노트북 원본은 이 최종 실행 폴더의 mlp_training_before.ipynb에 보존했습니다.

In [7]:
print((OUT/'environment.json').read_text(encoding='utf-8'))
print((OUT/'protected_verification_before_notebook.json').read_text(encoding='utf-8'))


{
  "python": "3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]",
  "libraries": {
    "scikit-learn": "1.9.1",
    "numpy": "2.5.4",
    "scipy": "1.18.1",
    "pandas": "3.1.0",
    "pyarrow": "26.0.1",
    "joblib": "1.6.0",
    "PyYAML": "6.0.3",
    "lightgbm": "4.7.0",
    "threadpoolctl": "3.7.0"
  },
  "blas_threads": 1
}
{
  "files": 838,
  "changed": []
}
